# DAAT — Validation and Robustness Analyses

Reproduces the validation analyses reported in the paper: effective sample
size, leave-one-scene-out cross-validation of the density rule, paired
significance testing, audit-log schema and re-derivation checks, the
deployment-observable event taxonomy, and storage accounting.

**Data.** Set `DRIVE_HINT` in the configuration cell to the folder
containing your `TEST/` results directory. Tracker outputs, audit logs and
affinity matrices are not distributed with this repository; regenerate them
with the tracking code, or point `DRIVE_HINT` at your own run directory.

**Sections 9-11** additionally require MOT17 ground truth (`GT_ROOT`) and raw
detections (`DET_ROOT`); they are skipped when those paths are unset.


## 0. Configuration — direct paths, then a local cache

**Do not recursively glob a mounted Drive.** Every `stat()` on Drive FUSE is a network
round-trip, so `glob("**", recursive=True)` over a 1.5 GB folder can run for hours.
This version uses the known folder structure directly, then copies the small files
(csv/json/txt only, no zips) to Colab local disk so every later cell runs at SSD speed.

Runtime: **use a plain CPU runtime.** Nothing here needs a GPU.

In [ ]:
import os, glob, re, json, csv, collections, itertools, sys, time, shutil
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, fisher_exact
from scipy.optimize import linear_sum_assignment
pd.set_option("display.width", 220)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.ismount("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# ---- EDIT THIS ONE LINE if your folder moves -------------------------------
DRIVE_HINT = "/content/drive/MyDrive/MOT-17"
# ----------------------------------------------------------------------------

# Shared Drives mount as 'Shareddrives' or 'Shared drives' depending on build.
if not os.path.isdir(DRIVE_HINT):
    for alt in [DRIVE_HINT.replace("MyDrive", "Shareddrives"),
                DRIVE_HINT.replace("MyDrive", "Shared drives")]:
        if os.path.isdir(alt): DRIVE_HINT = alt; break

DRIVE_TEST = os.path.join(DRIVE_HINT, "TEST")

# ---- auto-detect if the hint is wrong -------------------------------------
# Walks DIRECTORY NAMES ONLY (no file stats), depth-capped, stopping at the
# first folder named TEST that holds Test-* run directories. Keeps the notebook
# portable: nobody has to hard-code a personal Drive path.
def find_test_root(roots=("/content/drive",), max_depth=10, tick=5.0):
    """Collect ALL candidate TEST folders and return the richest one.
    Returning the first match is unsafe: a Drive can hold several folders
    named TEST (old copies, partial syncs, MOT17/test), and picking the wrong
    one silently breaks every downstream section."""
    import time as _t
    t0 = last = _t.time(); seen = 0; cands = []
    for root in roots:
        if not os.path.isdir(root): continue
        base = os.path.abspath(root).count(os.sep)
        for cur, dirs, _ in os.walk(root):
            seen += 1
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            if os.path.basename(cur) == "TEST":
                n_runs = sum(1 for d in dirs
                             if d.startswith("Test-") or d.startswith("SUBMISSION"))
                if n_runs: cands.append((n_runs, cur))
            if cur.count(os.sep) - base >= max_depth:
                dirs[:] = []; continue
            if _t.time() - last > tick:
                print(f"   [{_t.time()-t0:5.0f}s] scanned {seen} dirs, "
                      f"{len(cands)} candidates ...", flush=True)
                last = _t.time()
    if not cands:
        print(f"   no TEST folder found ({seen} dirs, {_t.time()-t0:.0f}s)")
        return None
    cands.sort(reverse=True)
    print(f"   {len(cands)} candidate TEST folder(s) after {_t.time()-t0:.0f}s:")
    for n, p in cands[:5]:
        print(f"      {n:3} run dirs  {p}")
    print(f"   -> using the richest: {cands[0][1]}")
    return cands[0][1]

if not os.path.isdir(DRIVE_TEST):
    print(f"'{DRIVE_TEST}' not found -- searching Drive for a TEST folder ...", flush=True)
    hit = find_test_root()
    if hit:
        DRIVE_TEST = hit
        DRIVE_HINT = os.path.dirname(hit)
    else:
        print("   no TEST folder found. Edit DRIVE_HINT above to the folder "
              "that CONTAINS your TEST directory.")

print("DRIVE_HINT :", DRIVE_HINT, "|", os.path.isdir(DRIVE_HINT))
print("DRIVE_TEST :", DRIVE_TEST, "|", os.path.isdir(DRIVE_TEST))
if os.path.isdir(DRIVE_TEST):
    print("run dirs   :", sorted(os.listdir(DRIVE_TEST))[:12])

# GT lives beside TEST in most layouts; auto-fill GT_ROOT if it is still unset.
def find_gt_root(near, max_depth=8):
    for cur, dirs, _ in os.walk(near):
        if cur.count(os.sep) - os.path.abspath(near).count(os.sep) >= max_depth:
            dirs[:] = []; continue
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if os.path.basename(cur) in ("train", "MOT17-train"):
            for d in dirs:
                if os.path.exists(os.path.join(cur, d, "gt", "gt.txt")): return cur
    return None

### Copy to local disk (run once, ~2–4 min)

Copies only `.csv`, `.json`, `.txt` — skips the `.zip` submissions and notebooks that make
up most of the 1.5 GB. Set `USE_LOCAL_CACHE = False` to read straight from Drive (slow).

In [ ]:
USE_LOCAL_CACHE = True
CACHE = "/content/sag_cache/TEST"
KEEP  = (".csv", ".json", ".txt")

def sync_from_drive(src, dst, keep=KEEP, tick=3.0):
    # Copy small files off Drive with a HEARTBEAT every `tick` seconds.
    # Drive FUSE is slow; silence here means work, not a hang.
    os.makedirs(dst, exist_ok=True)
    n = nbytes = ndirs = nskip = 0
    t0 = last = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] walking {src} ...", flush=True)
    for root, dirs, files in os.walk(src):
        ndirs += 1
        rel = os.path.relpath(root, src)
        out = os.path.join(dst, rel) if rel != "." else dst
        wanted = [f for f in files if f.lower().endswith(keep)]
        if wanted: os.makedirs(out, exist_ok=True)
        # heartbeat on entering each directory, even if nothing to copy
        if time.time() - last > tick:
            print(f"  [{time.time()-t0:6.0f}s] dirs={ndirs:<5} copied={n:<6} "
                  f"{nbytes/1e6:7.1f} MB  <- {rel[:58]}", flush=True)
            last = time.time()
        for f in wanted:
            d = os.path.join(out, f)
            if os.path.exists(d):
                nskip += 1; continue
            try:
                shutil.copy2(os.path.join(root, f), d)
                n += 1; nbytes += os.path.getsize(d)
            except Exception as e:
                print("  skip", f, e, flush=True)
            if time.time() - last > tick:
                print(f"  [{time.time()-t0:6.0f}s] dirs={ndirs:<5} copied={n:<6} "
                      f"{nbytes/1e6:7.1f} MB  <- {rel[:58]}", flush=True)
                last = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] DONE: {n} copied, {nskip} already cached, "
          f"{nbytes/1e6:.1f} MB, {ndirs} dirs, {time.time()-t0:.0f}s", flush=True)
    return dst

if USE_LOCAL_CACHE and os.path.isdir(DRIVE_TEST):
    TEST_ROOT = sync_from_drive(DRIVE_TEST, CACHE)
else:
    TEST_ROOT = DRIVE_TEST
print("TEST_ROOT =", TEST_ROOT)

### Resolve the specific files (bounded, with zip fallback)

In [ ]:
def bounded_find(root, filename, max_depth=8, verbose=False):
    # Depth-limited walk. Never recurses without a bound; returns first match.
    if not root or not os.path.isdir(root):
        if verbose: print(f"      (skip: {root} is not a directory)")
        return None
    root=os.path.abspath(root); base=root.rstrip(os.sep).count(os.sep); n=0
    for cur, dirs, files in os.walk(root):
        n+=1
        if cur.count(os.sep)-base >= max_depth: dirs[:]=[]
        if filename in files: return os.path.join(cur, filename)
    if verbose: print(f"      (searched {n} dirs under {root}, no {filename})")
    return None

WANT = ["perseq_rule.json", "sweep_results.csv"]
t0=time.time()
print("searching local cache ...", flush=True)
found = {w: bounded_find(TEST_ROOT, w, verbose=True) for w in WANT}
for w,p in found.items(): print(f"  {w:<22}{p}")

# fallback 1: search the whole Drive folder (not just TEST), bounded
if any(v is None for v in found.values()) and os.path.isdir(DRIVE_HINT):
    print("\nnot in cache -> searching DRIVE_HINT (bounded, may take a minute) ...", flush=True)
    for w in WANT:
        if found[w] is None:
            found[w] = bounded_find(DRIVE_HINT, w, max_depth=8, verbose=True)
            if found[w] is None and os.path.isdir(DRIVE_TEST):
                found[w] = bounded_find(DRIVE_TEST, w, max_depth=8, verbose=True)
            if found[w] is None:
                _up = os.path.dirname(DRIVE_HINT)
                found[w] = bounded_find(_up, w, max_depth=6, verbose=True)
            print(f"  {w:<22}{found[w]}", flush=True)

# fallback 2: some run folders are still ZIPPED on Drive -- look inside them
if any(v is None for v in found.values()):
    import zipfile
    print("\nstill missing -> scanning .zip archives for the files ...", flush=True)
    zips=[]
    for root in {TEST_ROOT, DRIVE_HINT}:
        if root and os.path.isdir(root):
            for cur, dirs, files in os.walk(root):
                if cur.count(os.sep)-os.path.abspath(root).count(os.sep) >= 4: dirs[:]=[]
                zips += [os.path.join(cur,f) for f in files if f.lower().endswith(".zip")]
    print(f"  {len(zips)} archives to check", flush=True)
    EXDIR="/content/sag_cache/_unzipped" if IN_COLAB else os.path.join(os.path.dirname(TEST_ROOT),"_unzipped")
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                names=zf.namelist()
                hits=[n for n in names if os.path.basename(n) in WANT]
                if hits:
                    print(f"  FOUND in {os.path.basename(z)}: {hits}", flush=True)
                    zf.extractall(EXDIR)
                    for w in WANT:
                        if found[w] is None:
                            found[w]=bounded_find(EXDIR,w,max_depth=10)
        except Exception as e:
            pass
    for w,p in found.items(): print(f"  {w:<22}{p}")

RULE_JSON, SWEEP_CSV = found["perseq_rule.json"], found["sweep_results.csv"]
print(f"\nsearch total {time.time()-t0:.1f}s")

### If still missing — show me what IS there

In [ ]:
# Diagnostic: run ONLY if the cell above left RULE_JSON / SWEEP_CSV as None.
if TEST_ROOT is None or not os.path.isdir(TEST_ROOT):
    print("TEST_ROOT is not set or does not exist:", TEST_ROOT)
    print("Fix DRIVE_HINT in section 0 and re-run that cell, then this one.")
elif RULE_JSON is None or SWEEP_CSV is None:
    print("TEST_ROOT subdirectories:")
    for d in sorted(os.listdir(TEST_ROOT))[:40]: print("   ", d)
    print("\nAnything that looks like a rule/calibration file:")
    pats=("rule","sweep","calib","density")
    n=0
    for cur, dirs, files in os.walk(TEST_ROOT):
        for f in files:
            if any(p in f.lower() for p in pats) and f.lower().endswith((".json",".csv")):
                print("   ", os.path.relpath(os.path.join(cur,f), TEST_ROOT)); n+=1
                if n>40: break
        if n>40: break
    print("\nZips present (may contain the calibration run):")
    for cur, dirs, files in os.walk(TEST_ROOT):
        for f in files:
            if f.lower().endswith(".zip"): print("   ", os.path.relpath(os.path.join(cur,f), TEST_ROOT))
    print("\n>>> Paste the right paths in manually, e.g.:")
    print("    RULE_JSON = '/content/drive/.../perseq_rule.json'")
    print("    SWEEP_CSV = '/content/drive/.../sweep_results.csv'")
else:
    print("Both files resolved -- skip this cell.")

### Baseline run + preflight

In [ ]:
print("scanning for audit-log directories ...", flush=True)
cands=collections.Counter(); t1=last=time.time(); nd=0
for cur, dirs, files in os.walk(TEST_ROOT):
    nd+=1
    k=sum(1 for f in files if f.endswith("_ids_events.csv"))
    if k: cands[cur]=k
    if time.time()-last>3.0:
        print(f"  [{time.time()-t1:5.0f}s] {nd} dirs, {len(cands)} with logs", flush=True); last=time.time()
print(f"  [{time.time()-t1:5.1f}s] {nd} dirs, {len(cands)} contain audit logs")

BASE_RUN=None
for d,_ in cands.most_common():
    if "Test-6" in d: BASE_RUN=d; break
if BASE_RUN is None and cands: BASE_RUN=cands.most_common(1)[0][0]

PAPER_ROOT=None

# ---- SET THIS to unlock sections 9 & 10 (flag calibration, differential error) ----
# Folder CONTAINING the scene folders, each with gt/gt.txt inside.
# Scene folders may be MOT17-02 or MOT17-02-DPM/-FRCNN/-SDP; both resolve.
GT_ROOT  = None   # e.g. ".../MOT17/train"  (folder holding <scene>/gt/gt.txt)
DET_ROOT = None   # raw YOLOX detections -> unlocks section 11

# If GT_ROOT is unset, look for a MOT17 train folder near the results.
if GT_ROOT is None:
    for _base in [DRIVE_HINT, os.path.dirname(DRIVE_HINT)]:
        if _base and os.path.isdir(_base):
            _g = find_gt_root(_base)
            if _g: GT_ROOT = _g; print("auto-detected GT_ROOT:", GT_ROOT); break
    if GT_ROOT is None:
        print("GT_ROOT not found automatically - set it manually to unlock sections 9-10.")
# -----------------------------------------------------------------------------
BASES=["MOT17-02","MOT17-04","MOT17-05","MOT17-09","MOT17-10","MOT17-11","MOT17-13"]
DETS=["DPM","FRCNN","SDP"]
print("BASE_RUN :", BASE_RUN, f"({cands.get(BASE_RUN,0)} audit logs)" if BASE_RUN else "")

checks={
 "RULE_JSON": RULE_JSON is not None and os.path.exists(RULE_JSON),
 "SWEEP_CSV": SWEEP_CSV is not None and os.path.exists(SWEEP_CSV),
 "TEST_ROOT": TEST_ROOT is not None and os.path.isdir(TEST_ROOT),
 "BASE_RUN":  BASE_RUN is not None,
}
opt={"PAPER_ROOT (sec 8)":PAPER_ROOT is not None,
     "GT_ROOT (sec 9,10)":GT_ROOT is not None,
     "DET_ROOT (sec 11)":DET_ROOT is not None}
for k,v in {**checks,**opt}.items(): print(f"  {'OK  ' if v else 'MISS'}  {k}")

CAN_RUN_RULE = checks["RULE_JSON"] and checks["SWEEP_CSV"]
CAN_RUN_LOGS = checks["TEST_ROOT"] and checks["BASE_RUN"]
print(f"\nSections 2,3 (rule + LOSO): {'YES' if CAN_RUN_RULE else 'NO -- fix paths above'}")
print(f"Sections 1,4,5,6,7 (logs)  : {'YES' if CAN_RUN_LOGS else 'NO'}")
print("\n(no SystemExit -- later cells guard themselves so you can run what works)")

## 1. Are the 21 "sequences" actually 21?

MOT17 ships 7 videos × 3 public detector sets. Under the **private** protocol you supply your
own YOLOX detections, so the three variants of a video should be identical runs.
If so, every statistic in the paper has n=7, not n=21.

In [ ]:
import hashlib
def md5(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(65536), b''): h.update(c)
    return h.hexdigest()

RUN = BASE_RUN
rows=[]
for b in BASES:
    for kind, tmpl in [("tracker output","{b}-{d}.txt"),
                       ("HOTA summary","{b}-{d}_hota_summary.csv")]:
        hs=[]
        for d in DETS:
            p=os.path.join(RUN, tmpl.format(b=b,d=d))
            hs.append(md5(p) if os.path.exists(p) else None)
        rows.append(dict(scene=b, artifact=kind,
                         identical=(len(set(hs))==1 and hs[0] is not None), md5=hs[0]))
trip=pd.DataFrame(rows)
display(trip)

n_ident = trip.identical.sum()
print(f"\n{n_ident}/{len(trip)} artifact groups identical across DPM/FRCNN/SDP")
if trip.identical.all():
    print("\n>>> CONFIRMED: triplicates are byte-identical.")
    print(">>> EFFECTIVE SAMPLE SIZE = 7 SCENES, not 21.")
    print(">>> Actions: fix abstract/§4.1 wording; run all paired tests at n=7;")
    print(">>>          remove the 'over 600 sequence-level evaluations' claim.")

## 2. Reconstruct the rule from the artifact and audit it against the paper

`perseq_rule.json` is the frozen rule you shipped. Check what it actually does versus
what §3.2 and Alg. 1 say it does.

In [ ]:
assert CAN_RUN_RULE, "RULE_JSON/SWEEP_CSV unresolved - see the locate cells in section 0"
rule = json.load(open(RULE_JSON))
print(json.dumps(rule, indent=2)[:1200])

df = pd.read_csv(SWEEP_CSV)
GRID  = sorted(df.track_thresh.unique())
dens  = df.groupby("base").density.first().to_dict()
H = {(r.base, round(r.track_thresh,3)): r.HOTA for r in df.itertuples()}
I = {(r.base, round(r.track_thresh,3)): r.IDF1 for r in df.itertuples()}
h  = lambda b,t: H[(b, round(t,3))]
bl = lambda b,t: 0.5*H[(b,round(t,3))] + 0.5*I[(b,round(t,3))]

print("\nGRID:", GRID)
print("selection_metric in artifact:", rule.get("selection_metric"))

findings=[]
if rule.get("selection_metric","").replace(" ","") != "HOTA":
    findings.append(f"Paper §3.2 / Alg.1 line 3 say 'HOTA-maximising'; artifact uses "
                    f"'{rule['selection_metric']}'.")
bt = rule["bucket_track_thresh"]
if any(round(v,3) not in [round(g,3) for g in GRID] for v in bt.values()):
    findings.append(f"Bucket thresholds {bt} are NOT grid points -> the rule averages "
                    f"per-sequence optima; it does not take a per-bucket argmax as described.")
opt = {b: max(GRID, key=lambda t: bl(b,t)) for b in dens}
if not any(abs(o-g)<1e-9 for o in opt.values() for g in [0.40,0.45]):
    findings.append("§4.3 says optima 'span the entire grid', but nothing optimises at 0.40/0.45.")
cuts=[rule["density_tertiles"]["low_below"], rule["density_tertiles"]["high_atabove"]]
on_pt=[c for c in cuts if any(abs(c-v)<1e-6 for v in dens.values())]
if on_pt:
    findings.append(f"Tertile cuts {on_pt} sit exactly on observed densities -> boundaries "
                    f"fitted to the sample (supports the 'memorising sequence identity' concern).")

print("\n--- PAPER vs ARTIFACT DISCREPANCIES ---")
for i,f in enumerate(findings,1): print(f"{i}. {f}")

print("\n--- does argmax choice change anything? ---")
for b in sorted(dens, key=lambda x: dens[x]):
    a,c = max(GRID,key=lambda t:h(b,t)), max(GRID,key=lambda t:bl(b,t))
    print(f"  {b}  d={dens[b]:6.2f}  argmaxHOTA={a}  argmaxBlend={c}  "
          f"{'agree' if a==c else '*** DIFFER ***'}")

## 3. Leave-one-scene-out CV, rule variants, and the oracle ceiling

**This answers rule cross-validation (no cross-validation), baseline choice (weak baseline) and baseline choice (best fixed
threshold) with one table.**

`oracle − best-global` is the hard ceiling: the most any density rule could ever gain.

In [ ]:
def buckets(train):
    d=sorted(dens[b] for b in train); n=len(d)
    lo,hi=d[n//3], d[(2*n)//3]
    return lambda x: "low" if x<lo else ("high" if x>=hi else "med")

def rule_mean_of_optima(train):        # what the CODE does
    bk=buckets(train); o={b:max(GRID,key=lambda t:bl(b,t)) for b in train}
    return bk,{k:float(np.mean([o[b] for b in train if bk(dens[b])==k]))
               for k in ["low","med","high"] if any(bk(dens[b])==k for b in train)}

def rule_bucket_argmax(train):         # what the PAPER describes
    bk=buckets(train); out={}
    for k in ["low","med","high"]:
        m=[b for b in train if bk(dens[b])==k]
        if m: out[k]=max(GRID,key=lambda t:np.mean([h(b,t) for b in m]))
    return bk,out

def rule_median_of_optima(train):
    bk=buckets(train); o={b:max(GRID,key=lambda t:bl(b,t)) for b in train}
    return bk,{k:float(np.median([o[b] for b in train if bk(dens[b])==k]))
               for k in ["low","med","high"] if any(bk(dens[b])==k for b in train)}

def interp(b,t):
    return float(np.interp(t, np.array(GRID,float), np.array([h(b,g) for g in GRID])))

def loso(fn, snap=False):
    out=[]
    for held in BASES:
        tr=[b for b in BASES if b!=held]
        bk,bt=fn(tr); tau=bt[bk(dens[held])]
        if snap: tau=min(GRID,key=lambda g:abs(g-tau))
        out.append(h(held,tau) if snap else interp(held,tau))
    return np.array(out)

base05 = np.array([h(b,0.5) for b in BASES])
base06 = np.array([h(b,0.6) for b in BASES])
oracle = np.array([max(h(b,t) for t in GRID) for b in BASES])
glob_hota = {t: np.mean([h(b,t) for b in BASES]) for t in GRID}
best_g    = max(GRID, key=lambda t: glob_hota[t])
print("mean HOTA by global tau:", {k:round(v,2) for k,v in glob_hota.items()},
      "\n -> best global tau =", best_g)

variants={
 "density rule, per-bucket argmax (as described)": loso(rule_bucket_argmax, snap=True),
 "density rule, mean-of-optima (as implemented)" : loso(rule_mean_of_optima),
 "density rule, mean-of-optima snapped to grid"  : loso(rule_mean_of_optima, snap=True),
 "density rule, median-of-optima snapped"        : loso(rule_median_of_optima, snap=True),
}
recs=[]
for name,v in variants.items():
    d=v-base06
    try: p=wilcoxon(v, base06).pvalue
    except Exception: p=np.nan
    recs.append(dict(method=name, HOTA=v.mean(), vs_tau06=d.mean(),
                     wins=f"{int((d>0).sum())}/7", p=p))
for name,v in [(f"fixed tau={best_g} (best global)",base06),
               ("fixed tau=0.50 (paper baseline)",base05),
               ("oracle per-scene (CEILING)",oracle)]:
    recs.append(dict(method=name, HOTA=v.mean(), vs_tau06=v.mean()-base06.mean(),
                     wins="-", p=np.nan))
tbl=pd.DataFrame(recs).sort_values("HOTA",ascending=False)
display(tbl.style.format({"HOTA":"{:.2f}","vs_tau06":"{:+.2f}","p":"{:.3f}"}))

print(f"\nORACLE CEILING over best global tau = {oracle.mean()-base06.mean():+.2f} HOTA")
print(f"scenes whose optimum is already {best_g}: "
      f"{sum(1 for b in BASES if max(GRID,key=lambda t:h(b,t))==best_g)}/7")
tbl.to_csv("loso_table.csv", index=False)

In [ ]:
# per-scene detail -> the figure to put in the paper
rowsd=[]
for held in BASES:
    tr=[b for b in BASES if b!=held]
    bk,bt=rule_mean_of_optima(tr); tau=bt[bk(dens[held])]
    gl={t:np.mean([h(b,t) for b in tr]) for t in GRID}
    bg=max(GRID,key=lambda t:gl[t])
    rowsd.append(dict(scene=held, density=dens[held], loso_tau=tau,
                      loso_HOTA=interp(held,tau), bestglobal_tau=bg,
                      bestglobal_HOTA=h(held,bg), tau06=h(held,0.6),
                      oracle=max(h(held,t) for t in GRID),
                      oracle_tau=max(GRID,key=lambda t:h(held,t))))
per=pd.DataFrame(rowsd)
per["delta_vs_global"]=per.loso_HOTA-per.bestglobal_HOTA
display(per.round(2))
per.to_csv("loso_perscene.csv", index=False)

## 4. Paired significance tests on the ablations (statistical resolution)

Table 4 invites readers to treat 0.04–0.12 HOTA gaps as real. Test them at the correct n=7.

In [ ]:
def load_perseq(metric="HOTA", detector="FRCNN"):
    runs=collections.defaultdict(dict)
    files=glob.glob(os.path.join(TEST_ROOT,"*","*","*_hota_summary.csv"))
    print(f"reading {len(files)} per-sequence metric files ...", flush=True)
    t0=last=time.time()
    for i,f in enumerate(files):
        if time.time()-last>3.0:
            print(f"  [{time.time()-t0:5.0f}s] {i}/{len(files)}", flush=True); last=time.time()
        parts=f.split(os.sep); run=parts[-3]+"/"+parts[-2]
        seq=os.path.basename(f).replace("_hota_summary.csv","")
        if not seq.endswith("-"+detector): continue
        r=list(csv.DictReader(open(f)))
        if r: runs[run][seq.replace("-"+detector,"")]=float(r[0][metric])*100
    print(f"  done in {time.time()-t0:.1f}s", flush=True)
    return {k:v for k,v in runs.items() if len(v)>=7}

runs=load_perseq()
print(f"{len(runs)} runs with >=7 scenes\n")
summ=pd.DataFrame([dict(run=k, n=len(v), mean=np.mean(list(v.values())),
                        std=np.std(list(v.values())))
                   for k,v in runs.items()]).sort_values("mean",ascending=False)
display(summ.round(2))

In [ ]:
BASELINE=os.path.relpath(BASE_RUN, TEST_ROOT).replace(os.sep,"/")  # discovered
COMPARISONS=[
 "Test-8-YOLOX/ALL-21-stage1-fusion",
 "Test-12-YOLOX/ALL-21-wsum-only",
 "Test-14-YOLOX-CLIPREID/ALL-21",
 "Test-13-YOLOX/ALL-21-v9-fasttracker",
 "Test-12-YOLOX/ALL-21-v8-visible",
 "Test-11-YOLOX/ALL-21-relax-gate",
 "Test-10-YOLOX/ALL-21-v6-whkalman",
]
base=runs[BASELINE]; scenes=sorted(base)
res=[]
for c in COMPARISONS:
    if c not in runs: print("missing:", c); continue
    v=runs[c]; s=[x for x in scenes if x in v]
    a=np.array([base[x] for x in s]); b=np.array([v[x] for x in s])
    d=b-a
    try: p=wilcoxon(b,a).pvalue
    except Exception: p=np.nan
    label=c.split("/")[-1]
    if label in ("ALL-21","ALL-21-stage1-fusion"): label=c.split("/")[0]+"/"+label
    res.append(dict(variant=label, n=len(s), delta=d.mean(),
                    sd=d.std(ddof=1), wins=f"{int((d>0).sum())}/{len(s)}", p=p))
if not res:
    print("No comparison runs found in TEST_ROOT -- section 4 needs the ablation runs.")
    R=pd.DataFrame(columns=["variant","n","delta","sd","wins","p","p_holm"])
else:
    R=pd.DataFrame(res).sort_values("p").reset_index(drop=True)
    # Holm-Bonferroni: step-down, monotonicity via MAXIMUM.accumulate
    m=len(R)
    R["p_holm"]=np.clip(np.maximum.accumulate((m-np.arange(m))*R.p.values), 0, 1)
    assert (R.p_holm.values >= R.p.values - 1e-12).all(), "Holm < raw p"
    assert (np.diff(R.p_holm.values) >= -1e-12).all(), "Holm not monotone"
display(R if R.empty else R.style.format(
    {"delta":"{:+.2f}","sd":"{:.2f}","p":"{:.3f}","p_holm":"{:.3f}"}))

MDD=float("nan")
if not R.empty:
    sd=R.sd.median(); MDD=1.13*sd
    print("\nMinimum detectable difference, paired n=7, power 0.8, alpha 0.05:")
    print(f"  pooled paired SD ~ {sd:.2f} HOTA  ->  MDD ~ {MDD:.2f} HOTA")
print("  => differences below this are NOT resolvable. State this in the Table 4 caption.")
R.to_csv("paired_tests.csv", index=False)

## 5. Does the audit log contain what §3.4 promises? (audit-log completeness)

§3.4 claims each record carries *"the relevant IoU-matrix rows"*. The event CSVs hold only
scalars — **but** each run also ships an `iou_matrices/` folder with per-frame
`frame_NNNNN_iou.csv` files containing the **full tracks x detections cost matrix**.

So the raw material for re-derivation exists. This section checks (a) what the event CSV
holds, (b) what the matrix files hold, and (c) whether a Hungarian assignment actually
re-derives from them.

In [ ]:
logs=glob.glob(os.path.join(TEST_ROOT,"*","*","*_ids_events.csv"))
schemas={tuple(pd.read_csv(f,nrows=0).columns) for f in logs[:400]}
cols=list(list(schemas)[0])
print(f"{len(logs)} event logs, {len(schemas)} distinct schema(s)")
print("event-CSV columns:", cols)

mat_dirs=[d for d in glob.glob(os.path.join(TEST_ROOT,"*","*","iou_matrices")) if os.path.isdir(d)]
mat_files=glob.glob(os.path.join(TEST_ROOT,"*","*","iou_matrices","frame_*_iou.csv"))
print(f"\niou_matrices dirs: {len(mat_dirs)} | matrix files: {len(mat_files)}")

CHECK={
 "full cost matrix (separate files)": len(mat_files)>0,
 "per-detection scores in header":    False,
 "track state + box per row":         False,
 "gating state":                      any("gate" in c.lower() for c in cols),
 "appearance feature vectors":        any("feat_vec" in c.lower() or "embedding" in c.lower()
                                          for c in cols),
 "matrices namespaced by sequence":   False,
}
if mat_files:
    d0=pd.read_csv(sorted(mat_files)[0], index_col=0)
    CHECK["per-detection scores in header"]=any("score=" in c for c in d0.columns)
    CHECK["track state + box per row"]=("track_state" in d0.columns and "track_box" in d0.columns)
    CHECK["matrices namespaced by sequence"]=any(
        re.search(r"MOT17-\d\d", os.path.basename(f)) for f in mat_files[:50])
print("\n--- §3.4 CLAIM AUDIT ---")
for k,v in CHECK.items(): print(f"  {'PRESENT' if v else 'ABSENT '}  {k}")

In [ ]:
# (c) Can a Hungarian assignment actually be re-derived from a logged matrix?
def parse_matrix(path, field="fused"):
    d=pd.read_csv(path, index_col=0)
    detcols=[c for c in d.columns if c.startswith("det")]
    def val(cell):
        m=re.search(rf"{field}=([\d.]+)", str(cell)); return float(m.group(1)) if m else 0.0
    C=np.array([[val(d.loc[t,c]) for c in detcols] for t in d.index])
    return list(d.index), detcols, C

MAT_DIR = os.path.join(BASE_RUN, "iou_matrices")
files = sorted(glob.glob(os.path.join(MAT_DIR, "frame_*_iou.csv")))
print(f"{len(files)} matrices in {os.path.relpath(MAT_DIR, TEST_ROOT)}")

if files:
    ok=0; tested=0; shapes=[]
    for f in files[:200]:
        try:
            tr, dt, C = parse_matrix(f)
            if C.size==0: continue
            r,c = linear_sum_assignment(-C)
            tested+=1; shapes.append(C.shape)
            if all(C[i,j]>=0 for i,j in zip(r,c)): ok+=1
        except Exception as e:
            print("  parse fail", os.path.basename(f), e)
    print(f"solver ran on {ok}/{tested} matrices; shapes e.g. {shapes[:5]}")
    tr,dt,C = parse_matrix(files[0])
    r,c = linear_sum_assignment(-C)
    print(f"\nexample {os.path.basename(files[0])}: {C.shape[0]} tracks x {C.shape[1]} dets")
    for i,j in list(zip(r,c))[:6]:
        if C[i,j]>0.1: print(f"   track {tr[i]} <- {dt[j].split('_')[0]}  fused={C[i,j]:.3f}")

In [ ]:
# What still blocks a court-grade re-derivation claim
print("VERDICT")
print("-"*70)
print("The full cost matrix IS logged -> §3.4's 'IoU-matrix rows' UNDERSTATES the artifact.")
print("Remaining gaps before 'faithful by construction' is defensible:\n")
gaps=[]
if not CHECK["matrices namespaced by sequence"]:
    gaps.append("Matrix files are named frame_NNNNN_iou.csv with NO sequence prefix, and one\n"
                "   iou_matrices/ folder serves all 21 pairs -> a given matrix cannot be\n"
                "   attributed to a sequence. FIX: namespace as <seq>_frame_NNNNN_iou.csv.")
if not CHECK["gating state"]:
    gaps.append("No gate mask is recorded, so a verifier cannot reproduce which entries were\n"
                "   suppressed before the solve. FIX: log the boolean gate mask alongside.")
frames_logged = len(files)
gaps.append(f"Matrices cover {frames_logged} frames (event frames only, with gaps), so the\n"
            "   claim must be scoped to logged events rather than the whole sequence.")
gaps.append("No verify.py ships with the artifact. FIX: include the re-derivation script and\n"
            "   report 'N of N logged events re-derive exactly' in the paper.")
for k,g in enumerate(gaps,1): print(f"{k}. {g}\n")

## 6. Deployment-observable event taxonomy (observable events)

R2 asks how DAAT decides an event is identity-relevant when no GT exists.
The logger already uses no GT — so this is a **naming** fix. Produce the taxonomy table.

In [ ]:
frames=[]
for f in glob.glob(os.path.join(BASE_RUN,"*-FRCNN_ids_events.csv")):
    d=pd.read_csv(f); d["scene"]=os.path.basename(f).split("_")[0]; frames.append(d)
ev=pd.concat(frames, ignore_index=True)
print("events:", len(ev), "| scenes:", ev.scene.nunique())

gt_like=[c for c in ev.columns if any(k in c.lower() for k in ("gt","truth","label","gtid"))]
print("GT-derived columns:", gt_like if gt_like else "NONE  ->  logger is GT-free  ->  observable events is terminology, not a bug")

ev["reason_class"]=ev.reason.astype(str).str.split().str[0]
ev["outcome"]=np.where(ev.reason_class.isin(["margin_failed","gate_failed",
                                             "registry_empty_or_immature"]),
                       "REFUSED (flagged)","ACCEPTED")
tax=(ev.groupby(["outcome","stage","reason_class"]).size()
       .reset_index(name="n").sort_values("n",ascending=False))
display(tax)
print("\n", ev.outcome.value_counts().to_string())
tax.to_csv("event_taxonomy.csv", index=False)

# schema fix: reason embeds numbers in free text -> split into real columns
ev["margin_val"]=ev.reason.astype(str).str.extract(r"margin=([\d.]+)").astype(float)
ev["thresh_val"]=ev.reason.astype(str).str.extract(r"<([\d.]+)").astype(float)
print("\nparsed margins:", ev.margin_val.notna().sum(),
      "-> SCHEMA FIX: emit reason/margin/threshold as separate columns.")

## 7. Storage budget (audit-log completeness) and 8. Anonymity scrub (anonymity)

In [ ]:
tot=collections.Counter(); nseq=set()
for f in glob.glob(os.path.join(BASE_RUN,"*.csv")):
    b=os.path.basename(f)
    for kind in ["ids_events","dlow_discards","registry","hota","mota"]:
        if kind in b: tot[kind]+=os.path.getsize(f); nseq.add(b.split("_")[0])
print(f"per-component totals over {len(nseq)} sequence-detector pairs:")
for k,v in tot.most_common(): print(f"  {k:<16}{v/1e6:8.2f} MB   ({v/1e6/max(len(nseq),1):.3f} MB/pair)")
print(f"  {'TOTAL':<16}{sum(tot.values())/1e6:8.2f} MB")
print("\nNOTE: adding full cost matrices at event frames will increase this.")
print("Recompute and report the breakdown; dedupe feature vectors by content hash.")

In [ ]:
PATTERNS={
 "submission id":      r"#\s?\d{6}",
 "method name+leaderboard": r"public MOT17 leaderboard",
 "submission date":    r"20\d\d-\d\d-\d\d",
 "named method":       r"DAAT",
 "anon repo link":     r"anonymous\.4open\.science\S+",
}
hits=[]
tex_files = glob.glob(os.path.join(PAPER_ROOT,"**","*.tex"), recursive=True) if PAPER_ROOT else []
for f in tex_files:
    for i,line in enumerate(open(f,encoding="utf-8",errors="ignore"),1):
        for name,pat in PATTERNS.items():
            if re.search(pat,line):
                hits.append(dict(file=os.path.relpath(f,PAPER_ROOT), line=i,
                                 issue=name, text=line.strip()[:100]))
A=pd.DataFrame(hits)
display(A if len(A) else "no .tex found -- set PAPER_ROOT (section 8 is optional)")
if len(A):
    print("\nFix: delete submission IDs + dates; remove the sentence tying the method")
    print("name to the public leaderboard; consider a placeholder name for review.")
    A.to_csv("anonymity_hits.csv", index=False)

---
# BLOCKED — need files not in the uploaded archive

These are the analyses **you still have to run**. Code is written; point the paths at Drive.

## 9. Flag calibration: base rate, precision, recall, ROC (flag calibration)

R1's sharpest empirical point. The 16.2% vs 2.7% comparison is confounded (flags fire at
crossings, which is where switches happen anyway) and **recall is never reported**.

Back-of-envelope from your own numbers: 3,837 flagged × 16.2% ≈ 622 flagged events
co-occurring with a true switch. Against ~1,800 total switches, recall ≈ **35%**.
Verify properly below.

**Needs:** `MOT17/train/<seq>/gt/gt.txt`

In [ ]:
def gt_path(seq, gt_root):
    """MOT17 ships each scene three times (…-DPM/-FRCNN/-SDP) with IDENTICAL gt.txt.
    Accept a bare scene name ('MOT17-02') or an explicit folder, and resolve it."""
    cands = [seq, f"{seq}-FRCNN", f"{seq}-DPM", f"{seq}-SDP"]
    for c in cands:
        p = os.path.join(gt_root, c, "gt", "gt.txt")
        if os.path.exists(p): return p
    raise FileNotFoundError(
        f"No gt.txt for {seq} under {gt_root}. Tried: {cands}. "
        f"GT_ROOT should be the folder CONTAINING the scene folders, "
        f"e.g. '.../MOT17/train' (not the gt.txt itself).")

def load_gt(seq, gt_root):
    g=pd.read_csv(gt_path(seq,gt_root),header=None,
                  names=["frame","id","x","y","w","h","conf","cls","vis"])
    return g[(g.conf==1)&(g.cls==1)]

def true_id_switches(pred_txt, gt_df, iou_thr=0.5):
    # Return set of (frame, gt_id) where the matched tracker id changes.
    pr=pd.read_csv(pred_txt,header=None,
                   names=["frame","id","x","y","w","h","conf","a","b","c"])
    def iou(A,B):
        xa=np.maximum(A[:,None,0],B[None,:,0]); ya=np.maximum(A[:,None,1],B[None,:,1])
        xb=np.minimum(A[:,None,0]+A[:,None,2],B[None,:,0]+B[None,:,2])
        yb=np.minimum(A[:,None,1]+A[:,None,3],B[None,:,1]+B[None,:,3])
        inter=np.clip(xb-xa,0,None)*np.clip(yb-ya,0,None)
        return inter/(A[:,None,2]*A[:,None,3]+B[None,:,2]*B[None,:,3]-inter+1e-9)
    last={}; sw=set()
    for fr in sorted(set(gt_df.frame)):
        G=gt_df[gt_df.frame==fr]; P=pr[pr.frame==fr]
        if len(G)==0 or len(P)==0: continue
        M=iou(G[["x","y","w","h"]].values, P[["x","y","w","h"]].values)
        r,c=linear_sum_assignment(-M)
        for i,j in zip(r,c):
            if M[i,j]<iou_thr: continue
            g=G.iloc[i].id; t=P.iloc[j].id
            if g in last and last[g]!=t: sw.add((fr,g))
            last[g]=t
    return sw

def calibrate_flags(seq, run_dir, gt_root, window=5):
    ev=pd.read_csv(os.path.join(run_dir,f"{seq}-FRCNN_ids_events.csv"))
    gt=load_gt(seq,gt_root)
    sw=true_id_switches(os.path.join(run_dir,f"{seq}-FRCNN.txt"), gt)
    sw_frames=np.array(sorted({f for f,_ in sw})) if sw else np.array([])
    ev["reason_class"]=ev.reason.astype(str).str.split().str[0]
    ev["flagged"]=ev.reason_class.isin(["margin_failed","gate_failed",
                                        "registry_empty_or_immature"])
    ev["near_switch"]=[bool(len(sw_frames)) and
                       np.any(np.abs(sw_frames-f)<=window) for f in ev.frame]
    ev["margin_val"]=ev.reason.astype(str).str.extract(r"margin=([\d.]+)").astype(float)
    return ev, len({f for f,_ in sw}), sw

if GT_ROOT is None:
    print("SET GT_ROOT to run. Then this produces, for flag calibration:")
    print("  (a) 2x2 contingency flagged x true-switch, restricted to CROSSING events")
    print("      -> the base-rate control R1 asked for (Fisher exact)")
    print("  (b) PRECISION, RECALL, F1 -- recall is the forensically critical number")
    print("  (c) ROC/AUC by sweeping the disambiguation margin delta")
else:
    allev=[]; tot_sw=0; sw_caught=0
    for b in BASES:
        e,n,sw = calibrate_flags(b, BASE_RUN, GT_ROOT)
        e["scene"]=b; allev.append(e); tot_sw += n
        # RECALL is over SWITCHES, not events: a switch counts as caught if at
        # least one FLAGGED event falls within the window. Counting flagged
        # events instead lets several flags near one switch inflate recall > 1.
        fl = e.loc[e.flagged, "frame"].values
        for f_sw in sorted({f for f, _ in sw}):
            if len(fl) and np.any(np.abs(fl - f_sw) <= 5): sw_caught += 1
    E=pd.concat(allev,ignore_index=True)
    ct=pd.crosstab(E.flagged, E.near_switch)
    display(ct); print("Fisher exact p =", fisher_exact(ct.values)[1])

    tp=int(((E.flagged)&(E.near_switch)).sum()); fp=int(((E.flagged)&(~E.near_switch)).sum())
    prec = tp/max(tp+fp,1)                 # of flags raised, how many sit near a real switch
    rec  = sw_caught/max(tot_sw,1)         # of real switches, how many raised a flag
    f1   = 2*prec*rec/max(prec+rec,1e-9)
    assert 0.0 <= rec <= 1.0, f"recall out of range ({rec}) - check switch counting"
    print(f"\nprecision = {prec:.3f}   ({tp} of {tp+fp} flags near a true switch)")
    print(f"recall    = {rec:.3f}   ({sw_caught} of {tot_sw} true switches flagged)")
    print(f"F1        = {f1:.3f}")
    print(f"\nSILENT ERRORS: {tot_sw-sw_caught} of {tot_sw} switches "
          f"({100*(1-rec):.1f}%) raised NO flag.")
    try:
        from sklearn.metrics import roc_auc_score
        m=E.dropna(subset=["margin_val"])
        if len(m) and m.near_switch.nunique()>1:
            print("AUC (margin as score):", round(roc_auc_score(m.near_switch, -m.margin_val),3))
    except ImportError:
        print("(sklearn unavailable - skipping AUC)")

## 10. Differential error analysis (differential error)

Daubert's "known error rates" prong is not satisfied by population-level accuracy.
MOT17 has no demographic annotation — but `gt.txt` **does** carry a visibility ratio,
and box height is a clean scale/distance proxy. Stratify by those and state the
demographic barrier explicitly.

**Needs:** `MOT17/train/<seq>/gt/gt.txt`

In [ ]:
def differential_error(seq, run_dir, gt_root):
    gt=load_gt(seq,gt_root)
    sw=true_id_switches(os.path.join(run_dir,f"{seq}-FRCNN.txt"), gt)
    sw_ids={i for _,i in sw}
    per=gt.groupby("id").agg(vis=("vis","mean"), height=("h","mean"),
                             n=("frame","size")).reset_index()
    per["switched"]=per.id.isin(sw_ids)
    per["vis_bin"]=pd.cut(per.vis,[0,.3,.6,.8,1.01],
                          labels=["<0.3","0.3-0.6","0.6-0.8",">0.8"])
    per["h_bin"]=pd.qcut(per.height,4,labels=["small","med-small","med-large","large"],
                         duplicates="drop")
    per["scene"]=seq
    return per

if GT_ROOT is None:
    print("SET GT_ROOT to run. Produces switch-rate stratified by:")
    print("  - GT visibility ratio (occlusion severity)")
    print("  - box height quartile (scale / distance from camera)")
    print("  - scene lighting (MOT17-02 night vs MOT17-04 day)")
    print("\nThen add to Limitations: MOT17 carries no demographic annotation,")
    print("so appearance/demographic-conditioned error is UNMEASURABLE here.")
    print("Naming the barrier is what the Daubert framing obliges you to do.")
else:
    P=pd.concat([differential_error(b, BASE_RUN, GT_ROOT) for b in BASES],
                ignore_index=True)
    for k in ["vis_bin","h_bin"]:
        t=P.groupby(k, observed=True).agg(n=("id","size"), switch_rate=("switched","mean"))
        print(f"\n--- switch rate by {k} ---"); print(t.round(3).to_string())
        lo,hi=t.switch_rate.min(),t.switch_rate.max()
        ratio = f"{hi/lo:.1f}x" if lo>0 else "n/a (a bin has zero switches)"
        print(f"  absolute spread: {hi-lo:.3f}   ratio: {ratio}")

## 11. Detector-contamination drift (detector contamination)

YOLOX-X saw the MOT17 training sequences, so scores are overconfident on *seen* footage.
That biases τ\* selection itself, not just absolute numbers. Show the direction of the bias.

**Needs:** raw detection files for train and test at a fixed probe threshold.

In [ ]:
if DET_ROOT is None:
    print("SET DET_ROOT to run. Produces:")
    print("  - detection-confidence distributions, TRAIN (seen) vs TEST (unseen)")
    print("  - KS test + percentile shift")
    print("\nIf test scores sit systematically LOWER, the fitted tau* is biased HIGH")
    print("and the deployed rule is CONSERVATIVE -- a finding in your favour, worth stating.")
    print("\nMitigation for future work: define tau as a PERCENTILE of the score")
    print("distribution rather than an absolute value, making the rule contamination-robust.")
else:
    from scipy.stats import ks_2samp
    def scores(split, seqs):
        out=[]
        for s in seqs:
            p=os.path.join(DET_ROOT,split,f"{s}.txt")
            if os.path.exists(p):
                out.append(pd.read_csv(p,header=None).iloc[:,6].values)
        return np.concatenate(out) if out else np.array([])
    tr=scores("train",BASES)
    te=scores("test",["MOT17-01","MOT17-03","MOT17-06","MOT17-07",
                      "MOT17-08","MOT17-12","MOT17-14"])
    if len(tr) and len(te):
        for q in [50,75,90,95]:
            print(f"  p{q}: train={np.percentile(tr,q):.3f}  test={np.percentile(te,q):.3f}"
                  f"  delta={np.percentile(te,q)-np.percentile(tr,q):+.3f}")
        print("KS:", ks_2samp(tr,te))

---
# 12. What still needs doing outside this notebook

**Cannot be computed — requires a decision or an external run:**

1. **Server submission of fixed τ=0.60.** §3 shows 0.60 is the best global threshold on
   training data and beats the LOSO rule. R2 asked for exactly this comparison. Check your
   MOTChallenge per-method submission quota before spending a slot.
2. **Re-run with cost-matrix logging enabled**, then report the verifier pass rate (§5).
   This is a code change plus one re-run of the submitted config.
3. **Confirm which split `sweep_results.csv` used.** It lives under a folder named
   `...-EASY` and its HOTA values (MOT17-04 = 87.7) do not match ALL-21 (65.6 mean).
   If the rule was fitted on one split and reported against another, that is a further
   disclosure item.
4. **Protocol-deviation paragraph** on the test-informed selection of the submitted
   configuration (protocol scope). A writing decision, not an analysis.
5. **Prose reframe of Contribution 1** around protocol legitimacy (baseline choice). Given §3,
   the accuracy claim is not recoverable.

---
# 13. Validation summary

Summary of all checks, populated **from the computed results above**.
Status fields are filled in by the analyses, so re-running regenerates them.

In [ ]:
# Pull computed evidence into variables the matrix can cite.
EV = {}
EV["n_eff"]        = 7 if trip.identical.all() else 21
EV["loso_delta06"] = float(tbl.loc[tbl.method.str.contains("as implemented"),"vs_tau06"].iloc[0])
EV["loso_argmax"]  = float(tbl.loc[tbl.method.str.contains("as described"),"vs_tau06"].iloc[0])
EV["ceiling"]      = float(oracle.mean()-base06.mean())
EV["best_global"]  = best_g
EV["mdd"]          = MDD
EV["n_discrep"]    = len(findings)
EV["mat_files"]    = len(mat_files) if "mat_files" in dir() else 0
EV["log_gaps"]     = len(gaps) if "gaps" in dir() else 0
EV["flagged"]      = int((ev.outcome=="REFUSED (flagged)").sum())
EV["accepted"]     = int((ev.outcome=="ACCEPTED").sum())
EV["gt_cols"]      = len(gt_like)
EV["storage_mb"]   = sum(tot.values())/1e6
EV["anon_hits"]    = len(A) if ("A" in dir() and isinstance(A,pd.DataFrame)) else 0
for k,v in EV.items(): print(f"  {k:<14}{v}")

In [ ]:
STATUS_RUNS   = "RUNS HERE"
STATUS_BLOCK  = "BLOCKED - needs data"
STATUS_DECIDE = "DECISION - not an analysis"

matrix = [
 ("audit-log completeness","Audit log cannot support re-derivation claim", STATUS_RUNS,
  f"full matrices ARE logged ({EV['mat_files']} files); {EV['log_gaps']} gaps remain (sec 5)",
  "Cite the matrices in 3.4; namespace by sequence; log gate mask; ship verify.py"),
 ("flag calibration","Flag calibration lacks base rate + recall", STATUS_BLOCK,
  f"{EV['flagged']} flagged / {EV['accepted']} accepted counted; GT needed for TP/FN",
  "Set GT_ROOT, run sec 9: 2x2 on crossings only, precision/recall/F1, ROC-AUC"),
 ("rule cross-validation","Density rule fit on 7 scenes, no CV", STATUS_RUNS,
  f"LOSO: rule {EV['loso_delta06']:+.2f} vs tau={EV['best_global']}; ceiling {EV['ceiling']:+.2f}",
  "Publish LOSO table; reframe Contribution 1 as protocol legitimacy"),
 ("protocol scope","Test-informed model selection", STATUS_DECIDE, "not computable",
  "Add 'Deviations from stated protocol' para; scope sec 3.2 claim to the rule only"),
 ("detector contamination","Detector contamination biases tau* selection", STATUS_BLOCK,
  "needs raw YOLOX detections, train + test",
  "Set DET_ROOT, run sec 11: score-distribution drift + KS test"),
 ("baseline choice","Weak baseline (tau=0.5) and small effect", STATUS_RUNS,
  f"tau={EV['best_global']} is best global and BEATS the rule by {-EV['loso_delta06']:.2f}",
  "Reframe Contribution 1; report LOSO honestly; tau=0.6 server run if quota allows"),
 ("statistical resolution","No paired statistics on ablations", STATUS_RUNS,
  f"n={EV['n_eff']} (not 21); MDD = {EV['mdd']:.2f} HOTA; nothing in Table 4 resolvable",
  "Report Wilcoxon + MDD in Table 4 caption"),
 ("differential error","No differential-error analysis", STATUS_BLOCK,
  "needs gt.txt visibility column",
  "Run sec 10: stratify by visibility + box height; state demographic barrier"),
 ("anonymity","Anonymity leak", STATUS_RUNS,
  f"{EV['anon_hits']} identifying strings across .tex files",
  "Strip submission IDs, dates, leaderboard sentence; consider placeholder name"),
 ("baseline choice","Compare vs best training-selected fixed threshold", STATUS_RUNS,
  f"best global = {EV['best_global']}; rule loses by {-EV['loso_delta06']:.2f}",
  "Same LOSO table answers rule cross-validation, baseline choice and baseline choice together"),
 ("observable events","Distinguish GT switch from observable risk event", STATUS_RUNS,
  f"{EV['gt_cols']} GT columns in logger -> terminology issue, not a bug",
  "Rename to 'identity-risk events'; publish taxonomy; confine GT to calibration"),
 ("effective sample size","Clarify 21 pairs vs 7 scenes", STATUS_RUNS,
  f"triplicates byte-identical -> n={EV['n_eff']}",
  "State '7 scenes x 3 detector labels'; remove 600-evaluations claim"),
]
M = pd.DataFrame(matrix, columns=["point","concern","status","computed_evidence","action"])
display(M)
M.to_csv("validation_summary.csv", index=False)
print("\n", M.status.value_counts().to_string())

In [ ]:
# Solvable / partially / not-solvable triage, derived from the matrix
def tier(row):
    if row.status==STATUS_RUNS:   return "1. SOLVED - evidence in hand"
    if row.status==STATUS_BLOCK:  return "2. SOLVABLE - run with your data"
    return "3. NOT AN ANALYSIS - editorial decision"
M["tier"]=M.apply(tier,axis=1)
for t in sorted(M.tier.unique()):
    sub=M[M.tier==t]
    print("="*78); print(t, f"({len(sub)} points)"); print("="*78)
    for r in sub.itertuples():
        print(f"  [{r.point}] {r.concern}")
        print(f"        evidence: {r.computed_evidence}")
        print(f"        action  : {r.action}")
    print()
M.to_csv("validation_status.csv", index=False)

# 14. Export every table to Markdown and LaTeX

Paste straight into the rebuttal or the paper.

In [ ]:
EXPORTS = {k:v for k,v in {
 "loso_comparison"   : tbl,
 "loso_perscene"     : per,
 "paired_tests"      : R,
 "event_taxonomy"    : tax,
 "triplicate_check"  : trip,
 "validation_summary"   : M,
}.items() if isinstance(v, pd.DataFrame) and not v.empty}
with open("all_tables.md","w") as f:
    for name,d in EXPORTS.items():
        f.write(f"\n\n## {name}\n\n")
        f.write(d.to_markdown(index=False))
print(open("all_tables.md").read()[:1500])

In [ ]:
# LaTeX for the two tables that belong in the paper
tex_loso = (tbl[["method","HOTA","vs_tau06","wins","p"]]
    .rename(columns={"method":"Threshold policy","vs_tau06":"$\\Delta$ vs $\\tau{=}0.6$",
                     "wins":"Wins","p":"$p$"})
    .to_latex(index=False, float_format="%.2f", escape=False,
              caption=("Leave-one-scene-out cross-validation of the density rule "
                       "($n{=}7$ scenes). The oracle row bounds the maximum achievable "
                       "gain of any density-to-threshold mapping on this data."),
              label="tab:loso"))
open("tab_loso.tex","w").write(tex_loso); print(tex_loso)

In [ ]:
tex_paired = "" if R.empty else (R[["variant","delta","sd","wins","p"]]
    .rename(columns={"variant":"Variant","delta":"$\\Delta$ HOTA","sd":"SD",
                     "wins":"Wins","p":"$p$"})
    .to_latex(index=False, float_format="%.2f", escape=False,
              caption=(f"Paired comparison against the v1 baseline over $n{{=}}7$ scenes "
                       f"(Wilcoxon signed-rank). The minimum detectable difference at this "
                       f"sample size is {1.13*R.sd.median():.2f} HOTA; no comparison in "
                       f"Table 4 is resolvable."),
              label="tab:paired"))
open("tab_paired.tex","w").write(tex_paired); print(tex_paired or "(skipped: no paired data)")
print("\nWrote: all_tables.md, tab_loso.tex, tab_paired.tex, validation_summary.csv, validation_status.csv")

---
# 15. Save every output to Drive

Colab's `/content/` is **ephemeral** — CSVs, `.tex` and `.md` written by the cells above
vanish when the runtime disconnects. This copies them into your Drive folder, timestamped
so re-runs never overwrite an earlier set.

The `.ipynb` itself autosaves to Drive if you opened it via *File > Open notebook > Drive*;
if you opened it from an upload, do **File > Save a copy in Drive** as well.

In [ ]:
import shutil, time, os, glob

STAMP   = time.strftime("%Y%m%d-%H%M%S")
OUT_DIR = os.path.join(DRIVE_HINT, "analysis_response", STAMP)
os.makedirs(OUT_DIR, exist_ok=True)

ARTIFACTS = ["loso_table.csv","loso_perscene.csv","paired_tests.csv",
             "event_taxonomy.csv","validation_summary.csv","validation_status.csv",
             "anonymity_hits.csv","all_tables.md","tab_loso.tex","tab_paired.tex"]

saved, missing = [], []
for f in ARTIFACTS:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(OUT_DIR, f))
        saved.append((f, os.path.getsize(f)))
    else:
        missing.append(f)

print(f"SAVED TO: {OUT_DIR}\n")
for f, sz in saved: print(f"  {f:<24}{sz/1024:8.1f} KB")
if missing:
    print("\nnot generated this run (section may have been skipped or blocked):")
    for f in missing: print("  ", f)

# verify they actually landed on Drive, not just locally
print("\nverifying on Drive ...")
on_drive = sorted(os.path.basename(p) for p in glob.glob(os.path.join(OUT_DIR, "*")))
print(f"  {len(on_drive)} files present: {on_drive}")
assert len(on_drive) == len(saved), "copy count mismatch - check Drive quota"
print("\nOK - outputs persisted.")

In [ ]:
# Optional: also snapshot this notebook itself next to the outputs.
try:
    from google.colab import _message
    nb_json = _message.blocking_request("get_ipynb", request="", timeout_sec=60)["ipynb"]
    import json as _json
    dest = os.path.join(OUT_DIR, f"DAAT_validation_analyses_{STAMP}.ipynb")
    with open(dest, "w") as fh: _json.dump(nb_json, fh)
    print("notebook snapshot saved:", dest)
except Exception as e:
    print("notebook snapshot unavailable:", e)
    print("Use File > Save a copy in Drive instead.")